# Python FastAPI & Pydantic — Complete Teaching Notebook
### From first HTTP request to a tested database-backed API
**Audience:** learners comfortable with Python functions, classes, exceptions, modules, and basic SQL. **Format:** 12 teaching sessions of 2–3 hours, including labs. **Python:** 3.11+. **Validation API:** Pydantic v2.

## Goal
Build, explain, test, and run APIs with FastAPI. Learn request/response contracts, Pydantic in depth, dependencies, persistence, authentication, authorization, asynchronous I/O, application lifecycle, testing, and delivery.

This notebook covers the core ecosystem comprehensively; no single notebook replaces every integration or production architecture. Advanced extension topics are explicitly labeled. All users, secrets, and data in demonstrations are synthetic.

### How to use this notebook
1. Install the packages in Setup, restart the kernel, then choose **Restart & Run All**.
2. Runnable examples use `TestClient` and in-process ASGI requests; no public server or account is needed.
3. Each small example has its own app, avoiding route collisions. Use context managers to close test clients and run lifespan events.
4. Terminal and deployment snippets are Markdown: read or copy them into a terminal, not a Python cell.
5. The capstone writes a runnable project into `fastapi_task_project/` beside the notebook and runs pytest. Rerunning replaces only the listed teaching files; do not edit those generated files without copying them first.
6. Never paste real secrets into notebook cells or commit `.env` files. Saved outputs are part of a notebook.

### Teaching sequence
| Session | Chapters | Outcome |
|---|---|---|
| 1 | 1–4 | Explain HTTP, ASGI, typing, and a first route |
| 2 | 5–10 | Handle path/query/header/cookie/body input |
| 3 | 11–17 | Build expressive Pydantic schemas and validators |
| 4 | 18–23 | Serialize safely, configure settings, migrate v1 code |
| 5 | 24–29 | Design responses, errors, routers, dependencies |
| 6 | 30–34 | Use async I/O, lifecycle, middleware, CORS |
| 7 | 35–39 | Handle forms/files/background tasks and realtime output |
| 8 | 40–44 | Use databases, transactions, and authentication |
| 9 | 45–48 | Test, debug, document, and version APIs |
| 10 | 49–51 | Deliver and operate an API responsibly |
| 11 | 52–54 | Run and understand the complete task API project |
| 12 | Labs and assessment | Extend, review, and demonstrate the project |

# Setup
Install in a fresh virtual environment or notebook kernel. The code follows Pydantic v2 and SQLAlchemy 2-style APIs. The final appendix records exact versions used to validate this notebook.

```bash
python -m venv .venv
# Linux/macOS:
source .venv/bin/activate
# Windows PowerShell:
# .venv\Scripts\Activate.ps1
python -m pip install "fastapi>=0.115,<1" "pydantic>=2.10,<3" \
  "pydantic-settings>=2,<3" "sqlalchemy>=2,<3" uvicorn httpx \
  python-multipart email-validator PyJWT "pwdlib[argon2]" pytest
python -m pip install jupyterlab nbformat nbclient nbconvert
jupyter lab
```

Inside a notebook you can use `%pip install ...` instead of the terminal command. Broad version ranges are for learning; use the tested exact versions or a reviewed lockfile for a reproducible project. `email-validator` enables `EmailStr`; `python-multipart` enables forms/uploads; `pwdlib[argon2]` provides password hashing.

In [1]:
import sys
import json
import asyncio
from datetime import date, datetime, timezone, timedelta
from decimal import Decimal
from enum import Enum
from typing import Annotated, Literal, Generic, TypeVar
from pathlib import Path
from uuid import UUID, uuid4
from importlib.metadata import version
from fastapi import (FastAPI, APIRouter, Query, Path as ApiPath, Body, Header, Cookie,
                     Depends, HTTPException, Response, Request, status, Form, File,
                     UploadFile, BackgroundTasks, WebSocket, WebSocketDisconnect)
from fastapi.testclient import TestClient
from fastapi.responses import JSONResponse, StreamingResponse, HTMLResponse, FileResponse
from pydantic import (BaseModel, Field, ConfigDict, ValidationError, EmailStr,
                      SecretStr, HttpUrl, TypeAdapter, RootModel, field_validator,
                      model_validator, field_serializer, computed_field, validate_call)
from pydantic_settings import BaseSettings, SettingsConfigDict
from IPython.display import display
print("Python", sys.version.split()[0])
for package in ["fastapi", "pydantic", "starlette", "httpx", "sqlalchemy"]:
    print(package, version(package))

Python 3.12.14
fastapi 0.141.1
pydantic 2.13.4
starlette 1.6.0
httpx 0.28.1
sqlalchemy 2.1.8


/opt/codex/runtimes/codex-primary-runtime/dependencies/python/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 1. HTTP and REST fundamentals

An API is a contract between a client and a service. A request has a method, URL, headers, and sometimes a body; a response has a status, headers, and sometimes a body. JSON is a representation, not a database.

| Method | Typical purpose | Idempotent by intended semantics? |
|---|---|---|
| GET | Read a resource | Yes |
| POST | Create or trigger an operation | Usually no |
| PUT | Replace a resource | Yes |
| PATCH | Partially update | Depends on the operation |
| DELETE | Remove a resource | Yes; later response status can differ |

Use nouns such as `/tasks/42`, meaningful status codes, and consistent error bodies. GET should not mutate application state. Retries of POST may need an idempotency key and durable deduplication.

**Status vocabulary:** 200 success, 201 created, 202 accepted for later work, 204 success without body; 400 malformed/business error by contract, 401 unauthenticated, 403 forbidden, 404 not found, 409 conflict, 422 validation failure, 429 rate limit, 500 unexpected server failure. Distinguish transport validation from business rules.

## 2. Framework pieces: FastAPI, Starlette, Pydantic, Uvicorn

FastAPI declares routes, dependencies, and API schemas. Starlette supplies ASGI/web primitives. Pydantic validates and serializes typed data. Uvicorn is an ASGI server. OpenAPI describes an API contract; Swagger UI and ReDoc display it.

Request flow: server → middleware → routing and dependency/input processing → endpoint/service → response handling → middleware → client. Cleanup and streaming timing depend on the resource and dependency scope; design for the complete resource lifetime.

Python type hints improve tooling, but normal Python does not enforce them at runtime. FastAPI/Pydantic use them to implement runtime contracts.

In [2]:
def greet(name: str) -> str:
    return f"Hello {name}"
print(greet("Aisha"))
print(greet(42))  # Python permits this; no runtime validator here.

Hello Aisha
Hello 42


## 3. First app and automatic documentation

A decorator registers a route. FastAPI uses the signature to infer inputs. Run this in process, then inspect the generated contract. `/docs`, `/redoc`, and `/openapi.json` are available by default; hiding documentation does not secure endpoints.

In [3]:
hello_app = FastAPI(title="Hello API", version="1.0.0")
@hello_app.get("/hello", tags=["basics"])
def hello(name: str = "World"):
    return {"message": f"Hello {name}"}

with TestClient(hello_app) as client:
    response = client.get("/hello", params={"name": "Aisha"})
    assert response.status_code == 200
    print(response.json())
    schema = client.get("/openapi.json").json()
    print("Documented paths:", list(schema["paths"]))
    assert client.get("/docs").status_code == 200

{'message': 'Hello Aisha'}
Documented paths: ['/hello']


## 4. Run a real server locally

Copy an app into `main.py`; do not run a blocking server inside this teaching notebook.

```python
from fastapi import FastAPI
app = FastAPI()
@app.get("/health")
def health():
    return {"status": "ok"}
```

```bash
python -m uvicorn main:app --reload --host 127.0.0.1 --port 8000
curl -i http://127.0.0.1:8000/health
```

`main:app` means the `app` object from module `main`. Reload is for development. If your code is in a package, use the dotted module path. Colab does not expose a local server to other people automatically; use the in-process examples for class.

## 5. Path parameters, types, enums, and route order

Path parameters identify a resource. Query parameters modify a read. Declare fixed paths before overlapping dynamic paths, such as `/users/me` before `/users/{user_id}`. Type validation does not rearrange route matching.

In [4]:
parameter_app = FastAPI()
class TaskState(str, Enum):
    todo = "todo"
    done = "done"

@parameter_app.get("/tasks/current")
def current_task():
    return {"id": 1}

@parameter_app.get("/tasks/{task_id}")
def read_task(task_id: Annotated[int, ApiPath(gt=0)], state: TaskState = TaskState.todo):
    return {"id": task_id, "state": state}

with TestClient(parameter_app) as client:
    assert client.get("/tasks/current").json() == {"id": 1}
    print(client.get("/tasks/3", params={"state": "done"}).json())
    assert client.get("/tasks/nope").status_code == 422
    assert client.get("/tasks/0").status_code == 422
    assert client.get("/tasks/1?state=unknown").status_code == 422

{'id': 3, 'state': 'done'}


## 6. Query parameters, bounds, lists, and pagination

Bound page size to avoid unbounded reads. A nullable parameter is optional only when it also has a default. Repeated query keys form lists. Offset pagination is simple; stable keyset/cursor pagination suits larger changing datasets.

In [5]:
query_app = FastAPI()
@query_app.get("/search")
def search(q: Annotated[str | None, Query(min_length=2, max_length=50)] = None,
           limit: Annotated[int, Query(ge=1, le=100)] = 20,
           offset: Annotated[int, Query(ge=0)] = 0,
           tags: Annotated[list[str] | None, Query()] = None):
    return {"q": q, "limit": limit, "offset": offset, "tags": tags or []}
with TestClient(query_app) as client:
    print(client.get("/search?tags=python&tags=api&limit=5").json())
    assert client.get("/search?limit=1000").status_code == 422
    assert client.get("/search?q=x").status_code == 422

{'q': None, 'limit': 5, 'offset': 0, 'tags': ['python', 'api']}


## 7. Headers, cookies, aliases, and source selection

Declare `Header`, `Cookie`, `Query`, `Path`, and `Body` explicitly when needed. Header names commonly use hyphens. Cookie authentication requires secure cookie attributes and a CSRF strategy; CORS is not a CSRF defense.

In [6]:
metadata_app = FastAPI()
@metadata_app.get("/metadata")
def metadata(x_request_id: Annotated[str | None, Header(max_length=80)] = None,
             session_hint: Annotated[str | None, Cookie()] = None):
    return {"request_id": x_request_id, "session_hint": session_hint}
with TestClient(metadata_app) as client:
    client.cookies.set("session_hint", "demo-only")
    print(client.get("/metadata", headers={"X-Request-ID": "abc-123"}).json())

{'request_id': 'abc-123', 'session_hint': 'demo-only'}


## 8. Request body and field constraints

A Pydantic model declares structured JSON input. Reject unexpected fields on write contracts when silent ignoring would hide mistakes. Validate string length and numeric ranges. Validation does not replace authorization or database constraints.

In [7]:
class ProductCreate(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)
    name: str = Field(min_length=1, max_length=80)
    price: Decimal = Field(gt=0, max_digits=10, decimal_places=2)
    stock: int = Field(ge=0)
    tags: list[str] = Field(default_factory=list, max_length=10)

body_app = FastAPI()
@body_app.post("/products", status_code=201, response_model=ProductCreate)
def create_product(product: ProductCreate):
    return product
with TestClient(body_app) as client:
    good = client.post("/products", json={"name": " Book ", "price": "12.50", "stock": 2})
    assert good.status_code == 201
    print(good.json())
    assert client.post("/products", json={"name": "", "price": -1, "stock": 0}).status_code == 422
    assert client.post("/products", json={"name": "A", "price": 1, "stock": 0, "admin": True}).status_code == 422

{'name': 'Book', 'price': '12.50', 'stock': 2, 'tags': []}


## 9. Multiple body fields and embedded models

With multiple body parameters, FastAPI expects an object keyed by parameter names. `Body(embed=True)` changes a single-model body to the same wrapper style. Request body shape is part of the public contract.

In [8]:
multi_app = FastAPI()
class Note(BaseModel):
    text: str = Field(min_length=1)
@multi_app.post("/notes")
def add_note(note: Note, importance: Annotated[int, Body(ge=1, le=5)]):
    return {"text": note.text, "importance": importance}
with TestClient(multi_app) as client:
    print(client.post("/notes", json={"note": {"text": "Read docs"}, "importance": 3}).json())
    assert client.post("/notes", json={"text": "Wrong shape", "importance": 3}).status_code == 422

{'text': 'Read docs', 'importance': 3}


## 10. Validation errors and required versus nullable

In Pydantic v2, `str | None` allows null but does not supply a default. Missing, explicit null, and empty string are different. FastAPI reports request validation failures as 422 by default. Avoid echoing credentials or whole request bodies in logs.

In [9]:
class PresenceDemo(BaseModel):
    required_nullable: str | None
    optional_nullable: str | None = None
try:
    PresenceDemo()
except ValidationError as exc:
    print([{ "field": e["loc"], "type": e["type"] } for e in exc.errors()])
assert PresenceDemo(required_nullable=None).required_nullable is None
with TestClient(body_app) as client:
    invalid = client.post("/products", json={})
    print([(e["loc"], e["type"]) for e in invalid.json()["detail"]])

[{'field': ('required_nullable',), 'type': 'missing'}]
[(['body', 'name'], 'missing'), (['body', 'price'], 'missing'), (['body', 'stock'], 'missing')]


# Pydantic v2 in Depth

## 11. Validation, coercion, and strict mode

Pydantic commonly converts compatible input values. Use strict validation when coercion would violate the contract. HTTP path/query values arrive as text, so strict numeric rules there require careful design. Python and JSON validation modes can have different strict behavior for types such as dates.

In [10]:
class CoercionDemo(BaseModel):
    quantity: int
print(CoercionDemo(quantity="3").model_dump())
class StrictQuantity(BaseModel):
    model_config = ConfigDict(strict=True)
    quantity: int
try:
    StrictQuantity(quantity="3")
except ValidationError:
    print("Strict mode rejected a string quantity")
assert StrictQuantity(quantity=3).quantity == 3

{'quantity': 3}
Strict mode rejected a string quantity


## 12. Nested models, collections, URLs, emails, and dates

Use nested models to express structure. `EmailStr` checks syntax and normalization, not inbox ownership. A syntactically valid URL is not safe to fetch automatically: outbound requests need destination policies. Dates and UUIDs serialize to JSON-friendly forms.

In [11]:
class Address(BaseModel):
    city: str
    postal_code: str  # Preserve leading zeros.
class CustomerProfile(BaseModel):
    id: UUID = Field(default_factory=uuid4)
    email: EmailStr
    website: HttpUrl | None = None
    joined: date
    address: Address
    interests: set[str] = Field(default_factory=set)
profile = CustomerProfile(email="learner@example.com", joined="2025-01-01",
                          address={"city": "Lahore", "postal_code": "00123"},
                          interests=["python", "python"])
print(profile.model_dump(mode="json"))
assert len(profile.interests) == 1

{'id': '7f1afbb2-fe02-4f39-b7d9-3e4af803fa60', 'email': 'learner@example.com', 'website': None, 'joined': '2025-01-01', 'address': {'city': 'Lahore', 'postal_code': '00123'}, 'interests': ['python']}


## 13. Field validators: before and after

A before validator sees raw input and should tolerate unexpected types until core validation runs. An after validator sees a parsed field. Return the validated value; raise `ValueError` for validation failures. Avoid assertions for user validation because optimized Python can disable them.

In [12]:
class Member(BaseModel):
    name: str = Field(min_length=2, max_length=50)
    @field_validator("name", mode="before")
    @classmethod
    def trim_name(cls, value):
        return value.strip() if isinstance(value, str) else value
    @field_validator("name")
    @classmethod
    def reject_reserved(cls, value: str) -> str:
        if value.casefold() == "admin":
            raise ValueError("Reserved display name")
        return value
print(Member(name="  Aisha  "))
try:
    Member(name="admin")
except ValidationError:
    print("Reserved value rejected")

name='Aisha'
Reserved value rejected


## 14. Cross-field rules and model validators

An after model validator checks relationships after parsing. Do not perform blocking database queries inside validators; use a service/dependency for existence or uniqueness checks, backed by database constraints.

In [13]:
class Booking(BaseModel):
    starts: date
    ends: date
    @model_validator(mode="after")
    def validate_range(self):
        if self.ends < self.starts:
            raise ValueError("ends must not be before starts")
        return self
print(Booking(starts="2025-01-01", ends="2025-01-03"))
try:
    Booking(starts="2025-01-03", ends="2025-01-01")
except ValidationError:
    print("Invalid date range rejected")

starts=datetime.date(2025, 1, 1) ends=datetime.date(2025, 1, 3)
Invalid date range rejected


## 15. Reusable annotated validators and advanced modes

Reusable annotations keep rules consistent. Before/wrap validators can control the pipeline; plain validators can bypass ordinary type validation, so use them carefully. For annotated validators, before/wrap ordering proceeds right-to-left and after ordering left-to-right.

In [14]:
from pydantic import AfterValidator

def not_blank(value: str) -> str:
    stripped = value.strip()
    if not stripped:
        raise ValueError("Must contain non-whitespace characters")
    return stripped
NonBlankText = Annotated[str, Field(max_length=80), AfterValidator(not_blank)]
class Label(BaseModel):
    text: NonBlankText
assert Label(text=" Python ").text == "Python"
try:
    Label(text="   ")
except ValidationError:
    print("Whitespace-only label rejected")

Whitespace-only label rejected


## 16. Discriminated unions

A discriminator chooses a schema using an explicit tag, producing a clearer contract than ambiguous unions. `Literal` constrains allowed tags. Use the same design for different event or payment shapes.

In [15]:
class EmailEvent(BaseModel):
    kind: Literal["email"]
    recipient: EmailStr
class SmsEvent(BaseModel):
    kind: Literal["sms"]
    phone: str = Field(pattern=r"^\+[1-9][0-9]{7,14}$")
Event = Annotated[EmailEvent | SmsEvent, Field(discriminator="kind")]
event_adapter = TypeAdapter(Event)
print(event_adapter.validate_python({"kind": "email", "recipient": "a@example.com"}))
try:
    event_adapter.validate_python({"kind": "push", "recipient": "a@example.com"})
except ValidationError:
    print("Unsupported discriminator rejected")

kind='email' recipient='a@example.com'
Unsupported discriminator rejected


## 17. Aliases and public contract names

Separate input names from internal Python names when integrating external formats. `validation_alias` changes accepted input; `serialization_alias` changes alias-mode output. Make alias policy explicit to avoid surprising clients.

In [16]:
class ExternalPerson(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    full_name: str = Field(validation_alias="fullName", serialization_alias="fullName")
person = ExternalPerson(fullName="Aisha Khan")
print(person.model_dump())
print(person.model_dump(by_alias=True))
assert ExternalPerson(full_name="Bilal").full_name == "Bilal"

{'full_name': 'Aisha Khan'}
{'fullName': 'Aisha Khan'}


## 18. Serialization, computed fields, and secrets

`model_dump()` returns Python values; `model_dump(mode="json")` returns JSON-compatible values; `model_dump_json()` returns JSON text. Serializers change representation, not validation. `SecretStr` masks displays but is not encryption. Separate public response schemas remain the strongest contract against accidental field exposure.

In [17]:
class InvoiceView(BaseModel):
    unit_price: Decimal
    quantity: int = Field(ge=1)
    secret_note: SecretStr = Field(exclude=True)
    @computed_field
    @property
    def total(self) -> Decimal:
        return self.unit_price * self.quantity
    @field_serializer("unit_price", "total", when_used="json")
    def money_as_text(self, value: Decimal):
        return f"{value:.2f}"
invoice = InvoiceView(unit_price="12.50", quantity=2, secret_note="synthetic")
print(invoice.model_dump_json())
assert "secret_note" not in invoice.model_dump()

{"unit_price":"12.50","quantity":2,"total":"25.00"}


## 19. Defaults, mutable collections, assignment, and immutability

Use `default_factory` for collections and dynamic values. Defaults are not validated by default unless configured. Assignment validation is optional. Frozen models prevent attribute rebinding but do not recursively freeze mutable nested objects.

In [18]:
class Preferences(BaseModel):
    model_config = ConfigDict(validate_assignment=True, validate_default=True)
    limit: int = Field(default=20, ge=1, le=100)
    tags: list[str] = Field(default_factory=list)
a, b = Preferences(), Preferences()
a.tags.append("python")
assert b.tags == []
try:
    a.limit = 0
except ValidationError:
    print("Invalid assignment rejected")

Invalid assignment rejected


## 20. TypeAdapter, RootModel, generics, and validated functions

Validate non-model containers with TypeAdapter. RootModel represents a single root value. Generic models avoid repeating response envelopes. `validate_call` validates function inputs outside FastAPI; return validation is separately configurable.

In [19]:
UserT = TypeVar("UserT")
class Page(BaseModel, Generic[UserT]):
    items: list[UserT]
    total: int = Field(ge=0)
class Tags(RootModel[list[str]]):
    pass
print(TypeAdapter(list[int]).validate_python(["1", 2]))
print(Tags(["python", "api"]).model_dump())
print(Page[Member](items=[Member(name="Aisha")], total=1).model_dump())
@validate_call
def multiply(a: int, b: int) -> int:
    return a * b
assert multiply("2", 3) == 6

[1, 2]
['python', 'api']
{'items': [{'name': 'Aisha'}], 'total': 1}


## 21. ORM objects, JSON Schema, and trusted construction

`from_attributes=True` reads attributes from objects such as ORM rows. `model_json_schema()` describes the contract. `model_construct()` bypasses validation, and `model_copy(update=...)` does not validate updates: avoid using either on untrusted request data.

In [20]:
class PublicName(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    name: str
class PlainObject:
    name = "Aisha"
    password_hash = "not-public"
print(PublicName.model_validate(PlainObject()).model_dump())
print("Schema keys:", list(ProductCreate.model_json_schema()))
unsafe_copy = CoercionDemo(quantity=2).model_copy(update={"quantity": "not-validated"})
assert isinstance(unsafe_copy.quantity, str)
print("Copy update skipped validation; validate merged data explicitly.")

{'name': 'Aisha'}
Schema keys: ['additionalProperties', 'properties', 'required', 'title', 'type']
Copy update skipped validation; validate merged data explicitly.


## 22. Pydantic Settings and environment configuration

Settings are a separate package in v2. Read deployment-supplied environment variables and secret files; do not hardcode production credentials. Cache settings per process when appropriate, and clear caches between tests. Default source precedence generally places constructor values ahead of environment, dotenv, secret files, and defaults; it can be customized.

In [21]:
import os
from unittest.mock import patch
class DemoSettings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="COURSE_", extra="ignore")
    app_name: str = "Teaching API"
    debug: bool = False
    page_size: int = Field(default=20, ge=1, le=100)
with patch.dict(os.environ, {"COURSE_PAGE_SIZE": "30"}):
    settings = DemoSettings()
    assert settings.page_size == 30
    print(settings.model_dump())

{'app_name': 'Teaching API', 'debug': False, 'page_size': 30}


## 23. Pydantic v1 → v2 migration reference

| Older pattern | Pydantic v2 pattern |
|---|---|
| `.dict()` / `.json()` | `.model_dump()` / `.model_dump_json()` |
| `parse_obj()` / `parse_raw()` for JSON | `model_validate()` / `model_validate_json()` |
| `class Config` | `model_config = ConfigDict(...)` |
| `orm_mode`, `from_orm()` | `from_attributes=True`, `model_validate()` |
| `@validator` | `@field_validator` |
| `@root_validator` | `@model_validator` |
| `BaseSettings` from pydantic | `BaseSettings` from pydantic_settings |
| Assuming Optional means a default | Add `= None` for an optional nullable field |

Migration requires behavior tests, not only mechanical renaming. Validators, union handling, aliases, settings, and serialization can change behavior. This notebook uses v2 throughout.

# API Design and Application Structure

## 24. Separate create, update, database, and public models

Write models accept only allowed fields. Public models omit internal state and credentials. Database models describe persistence, not necessarily the external API. Response model filtering helps enforce the public contract.

In [22]:
class AccountCreate(BaseModel):
    model_config = ConfigDict(extra="forbid")
    username: str
    password: str = Field(min_length=12, max_length=128)
class AccountPublic(BaseModel):
    id: int
    username: str
response_app = FastAPI()
@response_app.get("/account", response_model=AccountPublic)
def account():
    return {"id": 1, "username": "aisha", "password_hash": "synthetic-internal-value"}
with TestClient(response_app) as client:
    data = client.get("/account").json()
    assert "password_hash" not in data
    print(data)

{'id': 1, 'username': 'aisha'}


## 25. PUT, PATCH, omitted values, and explicit null

PATCH must distinguish fields the client omitted from fields explicitly provided as null. `exclude_unset=True` uses that distinction. Merge only allowed update fields and validate the final result; do not assume `model_copy(update=...)` revalidates it.

In [23]:
class TaskDraft(BaseModel):
    model_config = ConfigDict(extra="forbid")
    title: str = Field(min_length=1, max_length=80)
    description: str | None = None
class TaskPatch(BaseModel):
    model_config = ConfigDict(extra="forbid")
    title: str | None = Field(default=None, min_length=1, max_length=80)
    description: str | None = None
original = TaskDraft(title="Read", description="Chapter 1")
patch_data = TaskPatch(description=None)
changes = patch_data.model_dump(exclude_unset=True)
updated = TaskDraft.model_validate({**original.model_dump(), **changes})
assert updated.title == "Read" and updated.description is None
try:
    TaskDraft.model_validate({**original.model_dump(), **TaskPatch(title=None).model_dump(exclude_unset=True)})
except ValidationError:
    print("Explicit null title rejected by final model")
print(updated.model_dump())

Explicit null title rejected by final model
{'title': 'Read', 'description': None}


## 26. Status codes, headers, cookies, and response classes

204 must have no body. Use response classes for HTML, downloads, and streaming. Set secure cookie attributes for actual browser sessions. HTML must escape untrusted content. `FileResponse` should use server-controlled paths, never unchecked user paths.

In [24]:
formats_app = FastAPI()
@formats_app.delete("/items/{item_id}", status_code=204)
def remove_item(item_id: int):
    return Response(status_code=204)
@formats_app.get("/hello-page", response_class=HTMLResponse)
def hello_page():
    return "<h1>Hello, learner</h1>"
@formats_app.get("/headers")
def custom_headers(response: Response):
    response.headers["X-Course"] = "FastAPI"
    response.set_cookie("demo", "value", httponly=True, secure=True, samesite="lax")
    return {"ok": True}
with TestClient(formats_app) as client:
    result = client.delete("/items/1")
    assert result.status_code == 204 and result.content == b""
    assert "text/html" in client.get("/hello-page").headers["content-type"]
    assert client.get("/headers").headers["X-Course"] == "FastAPI"
    print("Status, HTML, headers, and cookie examples passed.")

Status, HTML, headers, and cookie examples passed.


## 27. HTTPException and safe exception handlers

Raise HTTPException for expected HTTP failures. A custom domain exception can be translated at the boundary. Unexpected exceptions should be logged with a correlation ID and return a safe generic response. Do not return tracebacks, SQL statements, tokens, or passwords.

In [25]:
errors_app = FastAPI()
class MissingTask(Exception):
    pass
@errors_app.exception_handler(MissingTask)
async def missing_task_handler(request: Request, exc: MissingTask):
    return JSONResponse(status_code=404, content={"code": "TASK_NOT_FOUND", "message": "Task not found"})
@errors_app.get("/tasks/{task_id}")
def error_task(task_id: int):
    raise MissingTask()
with TestClient(errors_app) as client:
    response = client.get("/tasks/999")
    assert response.status_code == 404
    print(response.json())

{'code': 'TASK_NOT_FOUND', 'message': 'Task not found'}


## 28. Depends, sub-dependencies, cleanup, and caching

Dependencies supply shared behavior without manual calls in every endpoint. They can depend on other dependencies. FastAPI normally caches a dependency result within a request; `use_cache=False` changes this. A yield dependency manages resources with try/finally. Choose cleanup timing deliberately when streaming or starting background work; do not hand request-scoped sessions to later jobs.

In [26]:
dependency_app = FastAPI()
def pagination(limit: Annotated[int, Query(ge=1, le=100)] = 10):
    return {"limit": limit}
def service_params(page: Annotated[dict, Depends(pagination)]):
    return page
@dependency_app.get("/items")
def dep_items(params: Annotated[dict, Depends(service_params)]):
    return params
with TestClient(dependency_app) as client:
    assert client.get("/items?limit=3").json() == {"limit": 3}
    dependency_app.dependency_overrides[pagination] = lambda: {"limit": 1}
    try:
        assert client.get("/items").json() == {"limit": 1}
    finally:
        dependency_app.dependency_overrides.clear()
print("Dependency injection and override passed.")

Dependency injection and override passed.


## 29. APIRouter and maintainable project structure

Use routers to group routes, tags, prefixes, and dependencies. Keep HTTP handling thin; use services for business rules and repositories/sessions for persistence. Avoid circular imports.

| Path | Responsibility |
|---|---|
| `app/main.py` | App factory, middleware, router registration |
| `app/api/tasks.py` | Task routes |
| `app/schemas/task.py` | Pydantic input/output contracts |
| `app/models/task.py` | Database models |
| `app/services/tasks.py` | Business rules |
| `app/db.py` | Engine/session management |
| `app/security.py`, `app/settings.py` | Authentication and configuration |
| `tests/` | Unit, integration, and authorization tests |

The capstone starts in one module so all moving parts are visible; splitting it by these responsibilities is a lab.

In [27]:
router_app = FastAPI()
v1_router = APIRouter(prefix="/api/v1", tags=["v1"])
@v1_router.get("/ping")
def ping():
    return {"pong": True}
router_app.include_router(v1_router)
with TestClient(router_app) as client:
    assert client.get("/api/v1/ping").json()["pong"] is True
print("Versioned router registered.")

Versioned router registered.


# Concurrency, Lifecycle, and Web Features

## 30. def versus async def: concurrency is not CPU parallelism

Use async endpoints with awaitable I/O libraries. A synchronous endpoint runs in a threadpool; a blocking call inside `async def` blocks the event loop unless offloaded. Normal helpers you call directly are not automatically offloaded. Use worker processes/jobs for CPU-heavy work. A shared mutable Python dictionary is not a production database, and each server worker has separate memory.

In [28]:
from starlette.concurrency import run_in_threadpool
async def wait_for_value(value):
    await asyncio.sleep(0.001)
    return value
async def concurrency_demo():
    results = await asyncio.gather(wait_for_value(1), wait_for_value(2))
    offloaded = await run_in_threadpool(lambda: sum([1, 2, 3]))
    return results, offloaded
# Notebook-safe top-level await; do not call asyncio.run() inside an active notebook loop.
results, offloaded = await concurrency_demo()
assert results == [1, 2] and offloaded == 6
print(results, offloaded)

[1, 2] 6


## 31. Outbound HTTP clients, timeouts, and retries

Reuse clients through lifespan rather than creating a client for each production request. Set timeouts, bound concurrency and response sizes, validate destinations, and handle upstream failures. Retry only selected transient failures with bounded backoff; non-idempotent operations need special care. This example mocks transport and never contacts a real website.

In [29]:
import httpx
async def upstream_demo():
    def mock_upstream(request):
        return httpx.Response(200, json={"status": "ok"})
    async with httpx.AsyncClient(transport=httpx.MockTransport(mock_upstream),
                                timeout=2.0, base_url="https://upstream.example") as client:
        response = await client.get("/status")
        response.raise_for_status()
        return response.json()
assert (await upstream_demo())["status"] == "ok"
print("Mock upstream request passed.")

Mock upstream request passed.


## 32. Lifespan startup and shutdown

Use an async context manager to acquire resources before serving and release them after shutdown. Examples include HTTP clients, connection pools, and model handles. Avoid expensive import-time work. Enter TestClient as a context manager to run lifespan.

In [30]:
from contextlib import asynccontextmanager
lifecycle_events = []
@asynccontextmanager
async def demo_lifespan(app: FastAPI):
    lifecycle_events.append("startup")
    app.state.ready = True
    try:
        yield
    finally:
        app.state.ready = False
        lifecycle_events.append("shutdown")
lifecycle_app = FastAPI(lifespan=demo_lifespan)
@lifecycle_app.get("/ready")
def ready(request: Request):
    return {"ready": request.app.state.ready}
with TestClient(lifecycle_app) as client:
    assert client.get("/ready").json()["ready"] is True
assert lifecycle_events == ["startup", "shutdown"]
print(lifecycle_events)

['startup', 'shutdown']


## 33. Middleware, request IDs, timing, and logging

Middleware can add response metadata around requests. Generate a server-owned correlation ID or validate inbound IDs before using them. Log structured fields such as route template, status, and duration; omit credentials and sensitive request payloads. Pure ASGI middleware is useful for more demanding streaming/context-variable behavior.

In [31]:
import time
middleware_app = FastAPI()
@middleware_app.middleware("http")
async def add_metadata(request: Request, call_next):
    request_id = str(uuid4())
    started = time.perf_counter()
    response = await call_next(request)
    response.headers["X-Request-ID"] = request_id
    response.headers["X-Process-Time-Ms"] = f"{(time.perf_counter() - started) * 1000:.2f}"
    return response
@middleware_app.get("/ping")
def middleware_ping():
    return {"ok": True}
with TestClient(middleware_app) as client:
    response = client.get("/ping")
    assert UUID(response.headers["X-Request-ID"])
    print("Timing header:", response.headers["X-Process-Time-Ms"])

Timing header: 0.41


## 34. CORS, browsers, and origin policy

An origin is scheme + host + port. CORS is a browser access policy, not authentication: curl and other clients are not blocked by it. Enumerate trusted origins. Credentialed CORS requires explicit settings; do not combine it casually with wildcard policies. Cookie-based state-changing endpoints also need CSRF protections.

In [32]:
from fastapi.middleware.cors import CORSMiddleware
cors_app = FastAPI()
cors_app.add_middleware(CORSMiddleware, allow_origins=["https://app.example.com"],
                        allow_credentials=True, allow_methods=["GET", "POST"],
                        allow_headers=["Authorization", "Content-Type"])
@cors_app.get("/data")
def cors_data():
    return {"ok": True}
with TestClient(cors_app) as client:
    allowed = client.get("/data", headers={"Origin": "https://app.example.com"})
    denied = client.get("/data", headers={"Origin": "https://other.example.com"})
    assert allowed.headers["access-control-allow-origin"] == "https://app.example.com"
    assert "access-control-allow-origin" not in denied.headers
print("CORS policy verified.")

CORS policy verified.


## 35. Forms and multipart requests

Use Form for URL-encoded form data and UploadFile for multipart files. A multipart request does not also carry an ordinary application/json body; encode structured metadata as a form field and validate it separately if needed. Never log login passwords.

In [33]:
forms_app = FastAPI()
@forms_app.post("/feedback")
def feedback(name: Annotated[str, Form(min_length=1, max_length=50)],
             score: Annotated[int, Form(ge=1, le=5)]):
    return {"name": name, "score": score}
with TestClient(forms_app) as client:
    result = client.post("/feedback", data={"name": "Aisha", "score": "5"})
    assert result.status_code == 200
    print(result.json())

{'name': 'Aisha', 'score': 5}


## 36. File uploads and resource limits

Read in bounded chunks. Treat filename and declared content type as untrusted metadata; inspect content using suitable parsers and server-generated storage keys. The endpoint cap below limits processed bytes, but the multipart parser may receive/spool data before the endpoint runs: also enforce request limits at the ingress/server layer. Add scanning and retention rules for production uploads.

In [34]:
uploads_app = FastAPI()
@uploads_app.post("/upload")
async def upload(file: Annotated[UploadFile, File()]):
    size = 0
    try:
        while chunk := await file.read(1024):
            size += len(chunk)
            if size > 4096:
                raise HTTPException(413, "File exceeds teaching limit")
        return {"bytes": size}
    finally:
        await file.close()
with TestClient(uploads_app) as client:
    assert client.post("/upload", files={"file": ("demo.txt", b"hello", "text/plain")}).json() == {"bytes": 5}
    assert client.post("/upload", files={"file": ("large.txt", b"x" * 5000)}).status_code == 413
print("Upload size handling passed.")

Upload size handling passed.


## 37. BackgroundTasks and durable job queues

BackgroundTasks runs after response handling in the same process; it is not a durable queue. A crash/restart can lose work. Use an external queue for expensive work, retries, delivery guarantees, and status tracking. Pass IDs and acquire fresh resources in jobs rather than reusing request-scoped database sessions.

In [35]:
jobs_app = FastAPI()
completed_jobs = []
def record_job(task_id: int):
    completed_jobs.append(task_id)
@jobs_app.post("/jobs", status_code=202)
def queue_job(background_tasks: BackgroundTasks):
    background_tasks.add_task(record_job, 1)
    return {"accepted": True}
with TestClient(jobs_app) as client:
    assert client.post("/jobs").status_code == 202
assert completed_jobs == [1]
print("TestClient waited for the demonstration background work.")

TestClient waited for the demonstration background work.


## 38. Streaming, downloads, and server-sent events

StreamingResponse sends chunks rather than building one response body. TestClient buffers the result, so this example tests content, not real network timing or disconnect behavior. Server-sent events use `text/event-stream` and framed events; browser/proxy buffering and reconnect behavior need separate testing. Download paths must be controlled by the server.

In [36]:
stream_app = FastAPI()
@stream_app.get("/export")
def export_rows():
    def chunks():
        yield "id,title\n"
        yield "1,Read docs\n"
    return StreamingResponse(chunks(), media_type="text/csv",
                             headers={"Content-Disposition": 'attachment; filename="tasks.csv"'})
with TestClient(stream_app) as client:
    response = client.get("/export")
    assert response.text == "id,title\n1,Read docs\n"
    print(response.text)

id,title
1,Read docs



## 39. WebSockets

WebSockets support bidirectional messages over a persistent connection. Authenticate the handshake, authorize every subscription/action, bound message sizes/rates, validate origin for browser clients, and handle disconnects. A process-local connection list does not broadcast across workers. This echo example demonstrates transport only.

In [37]:
websocket_app = FastAPI()
@websocket_app.websocket("/ws")
async def websocket_echo(websocket: WebSocket):
    await websocket.accept()
    try:
        while True:
            text = await websocket.receive_text()
            await websocket.send_json({"echo": text})
    except WebSocketDisconnect:
        pass
with TestClient(websocket_app) as client:
    with client.websocket_connect("/ws") as websocket:
        websocket.send_text("hello")
        assert websocket.receive_json() == {"echo": "hello"}
print("WebSocket round trip passed.")

WebSocket round trip passed.


# Persistence and Security

## 40. Database fundamentals: schema, SQL, and parameter binding

Use database constraints for durable invariants. Bind SQL parameters; never interpolate user input into SQL text. Pydantic checks request shape, not transaction isolation or concurrent uniqueness. Choose transactions around business operations, and avoid returning sensitive database fields.

In [38]:
import sqlite3
with sqlite3.connect(":memory:") as connection:
    connection.execute("CREATE TABLE notes (id INTEGER PRIMARY KEY, title TEXT NOT NULL)")
    connection.execute("INSERT INTO notes(title) VALUES (?)", ("Read docs",))
    untrusted_title = "' OR 1=1 --"
    rows = connection.execute("SELECT id, title FROM notes WHERE title = ?", (untrusted_title,)).fetchall()
    assert rows == []
    print(connection.execute("SELECT id, title FROM notes").fetchall())

[(1, 'Read docs')]


## 41. SQLAlchemy sessions, commits, rollback, and migrations

An engine owns connection-pool behavior. A Session tracks a unit of work; do not share one session across concurrent requests/tasks. Use a session per request/operation. Commit writes explicitly; rollback after failure. ORM relationship loading can cause N+1 queries; use appropriate eager loading and measure query counts.

The capstone uses synchronous SQLAlchemy in synchronous endpoints. For a fully asynchronous database stack, use an async driver, async engine/session, and awaitable operations end-to-end. SQLite is appropriate for this class; PostgreSQL introduces additional deployment/driver/transaction considerations.

`create_all()` creates missing tables but does not evolve existing schemas. Use versioned migrations (for example Alembic) for real schema changes. Test migrations and backup/restore procedures.

In [39]:
from sqlalchemy import create_engine, String, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session
class LessonBase(DeclarativeBase):
    pass
class LessonNote(LessonBase):
    __tablename__ = "lesson_notes"
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(80))
lesson_engine = create_engine("sqlite:///:memory:")
LessonBase.metadata.create_all(lesson_engine)
with Session(lesson_engine) as session:
    with session.begin():
        session.add(LessonNote(title="Transactions"))
    print(session.scalars(select(LessonNote.title)).all())
lesson_engine.dispose()

['Transactions']


## 42. Authentication versus authorization

Authentication asks who the caller is. Authorization asks whether that caller can perform this action on this resource. Checking a token is insufficient without owner/tenant/object checks. Return 401 with an appropriate challenge for missing/invalid bearer credentials; 403 for authenticated callers lacking permission. Some APIs intentionally use 404 for inaccessible objects to avoid revealing their existence.

| Mechanism | Typical use | Design concerns |
|---|---|---|
| API key | Service access | Storage, scope, rotation, revocation; not a person identity by itself |
| Cookie session | Browser application | Secure/HttpOnly/SameSite, session store, CSRF |
| Bearer access token | API authorization | Expiry, issuer, audience, signature, scope, revocation |
| OIDC/OAuth authorization code + PKCE | Login via identity provider | Provider integration and correct redirect/state/nonce handling |

The capstone includes a local password-login demonstration to teach primitives, not a general OAuth authorization server. For many deployed applications, use an established identity provider and authorization code + PKCE. Enforce HTTPS and rate-limit login/registration.

## 43. Password hashing and secret handling

Store password hashes using a password-hashing algorithm such as Argon2, not reversible encryption or plain SHA256. Verify with the library. Never return hashes or persist plaintext passwords. Production account systems also need recovery, abuse defenses, verification, and policy decisions.

In [40]:
from pwdlib import PasswordHash
password_hasher = PasswordHash.recommended()
synthetic_password = "Classroom-example-only-42!"
encoded_password = password_hasher.hash(synthetic_password)
assert password_hasher.verify(synthetic_password, encoded_password)
assert not password_hasher.verify("wrong-password", encoded_password)
print("Hash verification passed; neither password nor hash is printed.")

Hash verification passed; neither password nor hash is printed.


## 44. JWT claims, signature validation, and token expiry

A signed JWT is not encrypted; do not put secrets or sensitive profile data in it. Verify with a configured algorithm allowlist and expected issuer/audience; require necessary claims. Use short-lived access tokens. Refresh rotation, revocation, key rotation, and logout semantics need a fuller design. Decode errors should produce a safe authentication failure.

In [41]:
import secrets
import jwt
jwt_demo_key = secrets.token_urlsafe(48)
now = datetime.now(timezone.utc)
claims = {"sub": "user-1", "iss": "course-api", "aud": "course-client",
          "iat": now, "exp": now + timedelta(minutes=5)}
token = jwt.encode(claims, jwt_demo_key, algorithm="HS256")
decoded = jwt.decode(token, jwt_demo_key, algorithms=["HS256"], issuer="course-api",
                     audience="course-client", options={"require": ["sub", "iss", "aud", "iat", "exp"]})
assert decoded["sub"] == "user-1"
expired = jwt.encode({**claims, "exp": now - timedelta(seconds=10)}, jwt_demo_key, algorithm="HS256")
try:
    jwt.decode(expired, jwt_demo_key, algorithms=["HS256"], issuer="course-api", audience="course-client")
except jwt.ExpiredSignatureError:
    print("Expired token rejected.")
print("Valid token verified; token/key omitted from output.")

Expired token rejected.
Valid token verified; token/key omitted from output.


# Testing, Documentation, and Delivery

## 45. Testing strategy and dependency isolation

Use unit tests for pure rules, integration tests for database/HTTP behavior, and authorization tests for resource isolation. Verify happy paths, boundary values, invalid input, missing objects, duplicates, malformed tokens, expiry, and cross-user access. Fresh fixtures should isolate state. Test HTTP outcomes and invariants rather than private implementation details.

TestClient is synchronous. For async tests use an async ASGI client and manage lifespan explicitly; ASGI transport alone does not run lifecycle automatically. Network tests are still needed for timeouts, proxy behavior, TLS, streaming timing, and deployment configuration. The capstone runs pytest against a real temporary SQLite database.

In [42]:
def test_product_validation_contract():
    with TestClient(body_app) as client:
        assert client.post("/products", json={"name": "Book", "price": "1.20", "stock": 0}).status_code == 201
        assert client.post("/products", json={"name": "Book", "price": "0", "stock": 0}).status_code == 422
        assert client.post("/products", json={"name": "Book", "price": "1.234", "stock": 0}).status_code == 422

test_product_validation_contract()
print("Contract boundary tests passed.")

Contract boundary tests passed.


## 46. Async in-process request test

This app has no lifespan resources. When testing an app that does, enter its lifespan context before making requests. HTTPX ASGITransport avoids opening a network socket.

In [43]:
async def async_request_test():
    async with httpx.AsyncClient(transport=httpx.ASGITransport(app=hello_app), base_url="http://test") as client:
        response = await client.get("/hello", params={"name": "Async learner"})
        assert response.status_code == 200
        return response.json()
print(await async_request_test())

{'message': 'Hello Async learner'}


## 47. OpenAPI, examples, documentation, and compatibility

Add meaningful titles, descriptions, tags, response schemas, and examples; documentation should reflect real behavior. Preserve operation IDs if generated clients depend on them. A new required field, removed enum value, or changed response type can break clients. Decide an API versioning/deprecation policy; a URL prefix alone does not provide backward compatibility.

In [44]:
docs_app = FastAPI(title="Documented Tasks", version="1.0.0")
class DocumentedTask(BaseModel):
    title: str = Field(min_length=1, max_length=80, examples=["Read FastAPI docs"])
@docs_app.post("/tasks", response_model=DocumentedTask, status_code=201,
               summary="Create a task", tags=["tasks"],
               responses={409: {"description": "Business conflict, when applicable"}})
def documented_create(task: DocumentedTask):
    """Accept a task title and return its validated representation."""
    return task
contract = docs_app.openapi()
assert "201" in contract["paths"]["/tasks"]["post"]["responses"]
print("Schema models:", list(contract["components"]["schemas"]))

Schema models: ['DocumentedTask', 'HTTPValidationError', 'ValidationError']


## 48. Debugging and common mistakes

| Symptom | Likely cause | Next check |
|---|---|---|
| 422 | Wrong body shape, missing field, invalid query/path | Inspect `detail.loc` and the request schema |
| 404 on a known path | Prefix, route order, wrong server/module | Inspect registered paths and URL |
| 405 | Wrong HTTP method | Compare request method with route |
| 401 after login | Missing Bearer prefix, expired/wrong claims | Check token transport and verifier configuration |
| Browser fails, curl works | CORS/preflight or mixed content | Inspect browser network headers |
| SQLite locked | Long/concurrent writes | Shorten transactions; assess database choice |
| Missing lifespan resources in tests | Test client not entered | Use `with TestClient(app)` |
| Event loop already running | `asyncio.run` inside notebook | Use top-level `await` |
| Slow async endpoint | Blocking I/O or CPU work on loop | Profile and use appropriate execution model |
| Pydantic error after update | Null/missing contract or unvalidated merge | Check fields-set and validate final state |

Read tracebacks from the failure upward, reproduce with a minimal request, and add a regression test. Do not suppress errors with broad `except Exception: return {}`.

## 49. Deployment: processes, containers, configuration, and migrations

The capstone includes a teaching Dockerfile and health endpoints. Deployment snippets are provided for local practice; they are not executed or published by this notebook.

```bash
# From the generated project folder:
python -m uvicorn main:create_app --factory --host 127.0.0.1 --port 8000
# Multiple workers if the deployment/database design supports them:
# python -m uvicorn main:create_app --factory --host 0.0.0.0 --port 8000 --workers 2
```

Run migrations as a controlled deployment step, not once per worker at startup. Use an appropriate persistent database and volume, HTTPS ingress, trusted proxy configuration, deployment-supplied secrets, non-root containers, resource limits, dependency locks, backups, and tested rollback. Do not use reload in production. Each worker has independent memory; coordinate shared jobs/state using external systems.

Readiness asks whether an instance can serve requests; liveness asks whether it is alive. Design probes so temporary downstream trouble does not trigger endless restarts.

## 50. Observability, performance, and security review

Monitor request counts, error rates, latency percentiles, database latency/pool saturation, dependency failures, queue lag, and resource usage. Use logs/metrics/traces with correlation IDs. Avoid high-cardinality labels such as user IDs in metrics.

Bound pagination, upload sizes, timeouts, concurrency, and request rates. Test object/tenant isolation, output-field leakage, injection, unsafe file paths, SSRF, CORS/CSRF, secret storage, and trusted proxy behavior. Parameter binding prevents SQL text injection, not every possible database misuse. A valid request is not automatically a safe or authorized request.

Cache only with correct user/tenant keys and invalidation semantics. Measure first; avoid caching authorization failures or personalized data under shared keys.

## 51. Advanced extension map

| Topic | What to learn next | What to demonstrate |
|---|---|---|
| Async SQLAlchemy/PostgreSQL | Async sessions, transactions, pooling | Concurrent operations without shared sessions |
| Alembic | Schema revisions and rollout compatibility | Upgrade and rollback on a copy |
| OAuth scopes/RBAC/ABAC | Policy enforcement beyond identity | Role and object-level deny tests |
| OIDC provider integration | Authorization code + PKCE, JWKS rotation | Invalid issuer/audience/key rejection |
| Refresh tokens | Rotation, reuse detection, revocation | Old token reuse fails |
| Queues and scheduled jobs | Retries, idempotency, outbox | Crash/retry without duplicate side effect |
| SSE/WebSockets at scale | Broker, disconnects, backpressure | Multi-worker delivery test |
| Webhooks | Signatures, replay windows, retries | Tampered/replayed request rejection |
| Idempotency keys | Durable request deduplication | Retried POST creates one resource |
| Multi-tenancy | Tenant-scoped queries and constraints | Cross-tenant isolation |
| Generated clients | OpenAPI compatibility and code generation | Contract-change test |
| Property/load testing | Hypothesis, realistic load profiles | Invariants and measured capacity |

These are extension labs, not claims that the teaching app implements them.

# Complete Project: Authenticated Task API

## 52. Project brief and acceptance criteria

Build a personal task tracker with registration, login, and owner-scoped CRUD.

| Endpoint | Contract |
|---|---|
| `POST /users` | Register a username and password; 201; duplicate → 409 |
| `POST /token` | Form username/password; return short-lived bearer token |
| `GET /users/me` | Return public identity for a verified token |
| `POST /tasks` | Create an owned task; client cannot set owner/id |
| `GET /tasks` | Paginated, optionally filtered list of own tasks |
| `GET /tasks/{id}` | Return own task; absent/other owner's → 404 |
| `PATCH /tasks/{id}` | Update supplied fields; explicit null allowed only for description |
| `DELETE /tasks/{id}` | Delete own task; 204 without body |
| `GET /health/live`, `GET /health/ready` | Process and database probes |

**Persistence:** file-backed SQLite, SQLAlchemy per-request sessions, unique usernames, owner foreign keys. **Schemas:** separate request and response contracts, extra fields forbidden. **Security:** Argon2 hashes, signed/expiring JWTs with required issuer/audience/subject claims; database lookup of token subject; owner filtering on reads/writes.

**Teaching limits:** local password login, no refresh/logout/revocation service, no rate limiter/email verification/password reset, no migrations, no production identity provider, no durable jobs. `create_all` is for this classroom app. Do not treat it as a complete production identity system.

The next cells generate `main.py`, tests, exact-version requirements, Dockerfile, `.gitignore`, `.env.example`, and a README. Review the source before running a real server.

In [45]:
# This string is the complete generated main.py source.
# The following cell writes it to disk; edits here are included on rerun.
PROJECT_SOURCE = r'''
from contextlib import asynccontextmanager
from datetime import datetime, timedelta, timezone
from typing import Annotated, Literal

import jwt
from fastapi import FastAPI, Depends, HTTPException, Query, Response
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from pydantic import BaseModel, ConfigDict, Field, SecretStr, model_validator
from pydantic_settings import BaseSettings, SettingsConfigDict
from pwdlib import PasswordHash
from sqlalchemy import String, ForeignKey, create_engine, event, select, text
from sqlalchemy.exc import IntegrityError
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="TASK_API_", extra="ignore")
    database_url: str = "sqlite:///./tasks.db"
    jwt_secret: SecretStr = Field(min_length=32)
    jwt_issuer: str = "task-course-api"
    jwt_audience: str = "task-course-client"
    token_minutes: int = Field(default=15, ge=1, le=60)


class Base(DeclarativeBase):
    pass


class UserRow(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)
    username: Mapped[str] = mapped_column(String(40), unique=True, index=True)
    password_hash: Mapped[str] = mapped_column(String(512))


class TaskRow(Base):
    __tablename__ = "tasks"
    id: Mapped[int] = mapped_column(primary_key=True)
    owner_id: Mapped[int] = mapped_column(ForeignKey("users.id"), index=True)
    title: Mapped[str] = mapped_column(String(120))
    description: Mapped[str | None] = mapped_column(String(2000), nullable=True)
    state: Mapped[str] = mapped_column(String(20), default="todo")
    priority: Mapped[int] = mapped_column(default=2)


class UserCreate(BaseModel):
    model_config = ConfigDict(extra="forbid")
    username: str = Field(min_length=3, max_length=40, pattern=r"^[a-z][a-z0-9_]*$")
    password: SecretStr = Field(min_length=12, max_length=128)


class UserPublic(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    id: int
    username: str


class TokenPublic(BaseModel):
    access_token: str
    token_type: Literal["bearer"] = "bearer"


class TaskCreate(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)
    title: str = Field(min_length=1, max_length=120)
    description: str | None = Field(default=None, max_length=2000)
    state: Literal["todo", "doing", "done"] = "todo"
    priority: int = Field(default=2, ge=1, le=3)


class TaskUpdate(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)
    title: str | None = Field(default=None, min_length=1, max_length=120)
    description: str | None = Field(default=None, max_length=2000)
    state: Literal["todo", "doing", "done"] | None = None
    priority: int | None = Field(default=None, ge=1, le=3)

    @model_validator(mode="after")
    def reject_explicit_nulls(self):
        for name in ("title", "state", "priority"):
            if name in self.model_fields_set and getattr(self, name) is None:
                raise ValueError(f"{name} cannot be null")
        return self


class TaskPublic(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    id: int
    title: str
    description: str | None
    state: Literal["todo", "doing", "done"]
    priority: int


def create_app(settings: Settings | None = None) -> FastAPI:
    settings = settings if settings is not None else Settings()
    connect_args = {"check_same_thread": False} if settings.database_url.startswith("sqlite") else {}
    engine = create_engine(settings.database_url, connect_args=connect_args)
    if settings.database_url.startswith("sqlite"):
        @event.listens_for(engine, "connect")
        def enable_foreign_keys(dbapi_connection, connection_record):
            cursor = dbapi_connection.cursor()
            cursor.execute("PRAGMA foreign_keys=ON")
            cursor.close()

    hasher = PasswordHash.recommended()
    dummy_hash = hasher.hash("Classroom-dummy-password-not-an-account")
    bearer = OAuth2PasswordBearer(tokenUrl="token")

    @asynccontextmanager
    async def lifespan(app):
        # Classroom bootstrap only; use migrations before serving in deployment.
        Base.metadata.create_all(engine)
        try:
            yield
        finally:
            engine.dispose()

    app = FastAPI(title="Task Course API", version="1.0.0", lifespan=lifespan)

    def get_session():
        with Session(engine) as session:
            try:
                yield session
            except Exception:
                session.rollback()
                raise

    def credentials_error():
        return HTTPException(401, "Invalid authentication credentials",
                             headers={"WWW-Authenticate": "Bearer"})

    def current_user(token: Annotated[str, Depends(bearer)],
                     session: Annotated[Session, Depends(get_session)]):
        try:
            claims = jwt.decode(
                token, settings.jwt_secret.get_secret_value(), algorithms=["HS256"],
                issuer=settings.jwt_issuer, audience=settings.jwt_audience,
                options={"require": ["sub", "iss", "aud", "iat", "exp"]},
            )
            subject = claims["sub"]
            if not isinstance(subject, str) or not subject.isascii() or not subject.isdecimal():
                raise credentials_error()
            user_id = int(subject)
            if not 0 < user_id <= 2**63 - 1:
                raise credentials_error()
        except jwt.InvalidTokenError:
            raise credentials_error() from None
        user = session.get(UserRow, user_id)
        if user is None:
            raise credentials_error()
        return user

    def owned_task(task_id, user_id, session):
        task = session.scalar(select(TaskRow).where(TaskRow.id == task_id, TaskRow.owner_id == user_id))
        if task is None:
            raise HTTPException(404, "Task not found")
        return task

    @app.get("/health/live", tags=["health"])
    def live():
        return {"status": "alive"}

    @app.get("/health/ready", tags=["health"])
    def ready(session: Annotated[Session, Depends(get_session)]):
        session.execute(text("SELECT 1"))
        return {"status": "ready"}

    @app.post("/users", response_model=UserPublic, status_code=201, tags=["users"])
    def register(data: UserCreate, session: Annotated[Session, Depends(get_session)]):
        user = UserRow(username=data.username, password_hash=hasher.hash(data.password.get_secret_value()))
        session.add(user)
        try:
            session.commit()
        except IntegrityError:
            session.rollback()
            raise HTTPException(409, "Username unavailable") from None
        session.refresh(user)
        return user

    @app.post("/token", response_model=TokenPublic, tags=["auth"])
    def login(form: Annotated[OAuth2PasswordRequestForm, Depends()],
              session: Annotated[Session, Depends(get_session)]):
        user = session.scalar(select(UserRow).where(UserRow.username == form.username))
        # Both missing-user and bad-password cases perform password verification.
        stored_hash = user.password_hash if user is not None else dummy_hash
        valid = hasher.verify(form.password, stored_hash)
        if user is None or not valid:
            raise credentials_error()
        now = datetime.now(timezone.utc)
        token = jwt.encode({"sub": str(user.id), "iss": settings.jwt_issuer,
                            "aud": settings.jwt_audience, "iat": now,
                            "exp": now + timedelta(minutes=settings.token_minutes)},
                           settings.jwt_secret.get_secret_value(), algorithm="HS256")
        return TokenPublic(access_token=token)

    @app.get("/users/me", response_model=UserPublic, tags=["users"])
    def me(user: Annotated[UserRow, Depends(current_user)]):
        return user

    @app.post("/tasks", response_model=TaskPublic, status_code=201, tags=["tasks"])
    def add_task(data: TaskCreate, user: Annotated[UserRow, Depends(current_user)],
                 session: Annotated[Session, Depends(get_session)]):
        task = TaskRow(**data.model_dump(), owner_id=user.id)
        session.add(task)
        session.commit()
        session.refresh(task)
        return task

    @app.get("/tasks", response_model=list[TaskPublic], tags=["tasks"])
    def list_tasks(user: Annotated[UserRow, Depends(current_user)],
                   session: Annotated[Session, Depends(get_session)],
                   limit: Annotated[int, Query(ge=1, le=100)] = 20,
                   offset: Annotated[int, Query(ge=0)] = 0,
                   state: Literal["todo", "doing", "done"] | None = None):
        statement = select(TaskRow).where(TaskRow.owner_id == user.id)
        if state is not None:
            statement = statement.where(TaskRow.state == state)
        return session.scalars(statement.order_by(TaskRow.id).offset(offset).limit(limit)).all()

    @app.get("/tasks/{task_id}", response_model=TaskPublic, tags=["tasks"])
    def get_task(task_id: int, user: Annotated[UserRow, Depends(current_user)],
                 session: Annotated[Session, Depends(get_session)]):
        return owned_task(task_id, user.id, session)

    @app.patch("/tasks/{task_id}", response_model=TaskPublic, tags=["tasks"])
    def update_task(task_id: int, data: TaskUpdate, user: Annotated[UserRow, Depends(current_user)],
                    session: Annotated[Session, Depends(get_session)]):
        task = owned_task(task_id, user.id, session)
        for name, value in data.model_dump(exclude_unset=True).items():
            setattr(task, name, value)
        session.commit()
        session.refresh(task)
        return task

    @app.delete("/tasks/{task_id}", status_code=204, tags=["tasks"])
    def delete_task(task_id: int, user: Annotated[UserRow, Depends(current_user)],
                    session: Annotated[Session, Depends(get_session)]):
        task = owned_task(task_id, user.id, session)
        session.delete(task)
        session.commit()
        return Response(status_code=204)

    return app

'''

### Source reading guide
- `Settings` requires a secret; there is no fixed production fallback.
- `UserCreate`, `TaskCreate`, and `TaskUpdate` reject extra fields.
- `TaskUpdate.model_fields_set` distinguishes omission from explicit null.
- `get_session` closes/rolls back request resources. All database endpoints use `def`.
- `current_user` validates signature/claims, checks the subject format, and fetches the user.
- `owned_task` applies owner filtering for read, update, and delete.
- `register` catches the database uniqueness constraint instead of relying only on a pre-check.
- `login` performs dummy password verification for absent users; this is not a substitute for rate limiting.
- The database persists across app restarts. The tests create isolated temporary databases.

**Discussion:** Where would account disablement, roles, refresh revocation, migrations, and structured audit logs fit?

In [46]:
# Complete pytest suite; saved as test_api.py.
TEST_SOURCE = r'''
from datetime import datetime, timedelta, timezone
import secrets
import jwt
import pytest
from fastapi.testclient import TestClient
from main import Settings, create_app

PASSWORD = "Teaching-password-only-42!"

@pytest.fixture
def settings(tmp_path):
    return Settings(database_url=f"sqlite:///{tmp_path / 'test.db'}", jwt_secret=secrets.token_urlsafe(48))

@pytest.fixture
def client(settings):
    with TestClient(create_app(settings)) as client:
        yield client

def register(client, username):
    response = client.post("/users", json={"username": username, "password": PASSWORD})
    assert response.status_code == 201
    assert "password" not in response.json() and "password_hash" not in response.json()
    return response.json()

def login(client, username):
    response = client.post("/token", data={"username": username, "password": PASSWORD})
    assert response.status_code == 200
    return {"Authorization": "Bearer " + response.json()["access_token"]}

@pytest.fixture
def alice(client):
    register(client, "alice")
    return login(client, "alice")

def test_health_and_docs(client):
    assert client.get("/health/live").status_code == 200
    assert client.get("/health/ready").status_code == 200
    assert "/tasks" in client.get("/openapi.json").json()["paths"]

def test_registration_duplicate(client):
    register(client, "alice")
    duplicate = client.post("/users", json={"username": "alice", "password": PASSWORD})
    assert duplicate.status_code == 409

def test_login_errors(client):
    register(client, "alice")
    for username, password in [("alice", "wrong"), ("missing", PASSWORD)]:
        response = client.post("/token", data={"username": username, "password": password})
        assert response.status_code == 401
        assert response.headers["www-authenticate"] == "Bearer"

def test_public_identity(client, alice):
    assert client.get("/users/me", headers=alice).json()["username"] == "alice"
    assert client.get("/users/me").status_code == 401
    assert client.get("/users/me", headers={"Authorization": "Bearer invalid"}).status_code == 401

def test_crud_and_null_semantics(client, alice):
    response = client.post("/tasks", headers=alice, json={"title": " Read docs ", "description": "Chapter 1"})
    assert response.status_code == 201
    task = response.json()
    assert task["title"] == "Read docs"
    url = f"/tasks/{task['id']}"
    assert client.get(url, headers=alice).status_code == 200
    assert client.patch(url, headers=alice, json={"description": None}).json()["description"] is None
    assert client.patch(url, headers=alice, json={"title": None}).status_code == 422
    assert client.patch(url, headers=alice, json={"state": "done"}).json()["state"] == "done"
    assert len(client.get("/tasks?state=done", headers=alice).json()) == 1
    assert client.get("/tasks?state=todo", headers=alice).json() == []
    response = client.delete(url, headers=alice)
    assert response.status_code == 204 and response.content == b""
    assert client.get(url, headers=alice).status_code == 404

@pytest.mark.parametrize("payload", [{"title": ""}, {"title": "  "}, {"title": "x", "priority": 0},
                                    {"title": "x", "state": "bad"}, {"title": "x", "owner_id": 999}])
def test_invalid_task_payload(client, alice, payload):
    assert client.post("/tasks", headers=alice, json=payload).status_code == 422

def test_pagination_and_bounds(client, alice):
    for title in ["One", "Two", "Three"]:
        assert client.post("/tasks", headers=alice, json={"title": title}).status_code == 201
    assert len(client.get("/tasks?limit=2", headers=alice).json()) == 2
    assert client.get("/tasks?limit=2&offset=2", headers=alice).json()[0]["title"] == "Three"
    assert client.get("/tasks?limit=101", headers=alice).status_code == 422
    assert client.get("/tasks?offset=-1", headers=alice).status_code == 422

def test_owner_isolation(client, alice):
    created = client.post("/tasks", headers=alice, json={"title": "Private"}).json()
    register(client, "bob")
    bob = login(client, "bob")
    url = f"/tasks/{created['id']}"
    assert client.get("/tasks", headers=bob).json() == []
    assert client.get(url, headers=bob).status_code == 404
    assert client.patch(url, headers=bob, json={"title": "Stolen"}).status_code == 404
    assert client.delete(url, headers=bob).status_code == 404
    assert client.get(url, headers=alice).json()["title"] == "Private"

@pytest.mark.parametrize("change", ["expired", "audience", "issuer", "missing_exp", "unknown_user", "bad_subject", "wrong_key"])
def test_rejects_invalid_token_claims(client, settings, alice, change):
    user_id = client.get("/users/me", headers=alice).json()["id"]
    now = datetime.now(timezone.utc)
    claims = {"sub": str(user_id), "iss": settings.jwt_issuer, "aud": settings.jwt_audience,
              "iat": now, "exp": now + timedelta(minutes=5)}
    if change == "expired": claims["exp"] = now - timedelta(seconds=10)
    if change == "audience": claims["aud"] = "other-client"
    if change == "issuer": claims["iss"] = "other-issuer"
    if change == "missing_exp": claims.pop("exp")
    if change == "unknown_user": claims["sub"] = "99999"
    if change == "bad_subject": claims["sub"] = "not-an-id"
    key = secrets.token_urlsafe(48) if change == "wrong_key" else settings.jwt_secret.get_secret_value()
    token = jwt.encode(claims, key, algorithm="HS256")
    assert client.get("/users/me", headers={"Authorization": "Bearer " + token}).status_code == 401

def test_persistence_after_restart(settings):
    with TestClient(create_app(settings)) as first:
        register(first, "alice")
        headers = login(first, "alice")
        assert first.post("/tasks", headers=headers, json={"title": "Persisted"}).status_code == 201
    with TestClient(create_app(settings)) as second:
        headers = login(second, "alice")
        assert second.get("/tasks", headers=headers).json()[0]["title"] == "Persisted"

'''

## 53. Write the project and its supporting files

This cell writes the listed files under the current working directory. Exact versions come from the environment executing this notebook; the final appendix also records the originally tested versions. Avoid rerunning the generator over your own edited project without saving those edits elsewhere.

In [47]:
project_dir = Path("fastapi_task_project")
project_dir.mkdir(exist_ok=True)
(project_dir / "main.py").write_text(PROJECT_SOURCE, encoding="utf-8")
(project_dir / "test_api.py").write_text(TEST_SOURCE, encoding="utf-8")
packages = ["fastapi", "starlette", "pydantic", "pydantic-settings", "sqlalchemy", "uvicorn",
            "httpx", "python-multipart", "email-validator", "PyJWT", "pwdlib", "argon2-cffi", "pytest"]
requirements = "\n".join(f"{name}=={version(name)}" for name in packages) + "\n"
(project_dir / "requirements.txt").write_text(requirements, encoding="utf-8")
(project_dir / ".gitignore").write_text(".venv/\n.env\n*.db\n__pycache__/\n.pytest_cache/\n", encoding="utf-8")
(project_dir / ".env.example").write_text(
    "# Export these values in your shell; the app does not automatically load this file.\n"
    "TASK_API_DATABASE_URL=sqlite:///./tasks.db\n"
    "# TASK_API_JWT_SECRET must be a newly generated secret, at least 32 characters.\n", encoding="utf-8")
dockerfile = """FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
RUN useradd --create-home appuser && mkdir /data && chown appuser:appuser /data
COPY main.py .
USER appuser
ENV TASK_API_DATABASE_URL=sqlite:////data/tasks.db
EXPOSE 8000
CMD ["python", "-m", "uvicorn", "main:create_app", "--factory", "--host", "0.0.0.0", "--port", "8000"]
"""
(project_dir / "Dockerfile").write_text(dockerfile, encoding="utf-8")
(project_dir / ".dockerignore").write_text(".env\n*.db\n.venv\n__pycache__\n.pytest_cache\n", encoding="utf-8")
readme = """# Teaching Task API
Python 3.11+; Pydantic v2. Generated from the FastAPI teaching notebook.

## Local setup
Create and activate a virtual environment, then:
```bash
python -m pip install -r requirements.txt
python -m pytest -q
```
Generate a secret with `python -c "import secrets; print(secrets.token_urlsafe(48))"`.
Export that value as TASK_API_JWT_SECRET in your local shell (do not commit it).
Linux/macOS: `export TASK_API_JWT_SECRET='<your generated value>'`
PowerShell: `$env:TASK_API_JWT_SECRET='<your generated value>'`
Then:
```bash
python -m uvicorn main:create_app --factory --host 127.0.0.1 --port 8000
```
Visit http://127.0.0.1:8000/docs. Register at POST /users. Use Authorize with
that username/password; the token endpoint expects form fields.
The app reads real environment variables; .env.example is documentation only.
SQLite data lives in tasks.db by default. Do not commit it.

## Docker (optional local practice)
```bash
docker build -t task-course .
docker volume create task-course-data
docker run --rm -p 127.0.0.1:8000:8000 -e TASK_API_JWT_SECRET -v task-course-data:/data task-course
```
Supply a generated TASK_API_JWT_SECRET in your host environment first.
Container builds and live-server deployment are not verified by the notebook.

## Classroom scope
Password login, Argon2 hashes, verified short-lived JWTs, owner-scoped CRUD,
SQLite persistence, request validation, and isolated integration tests.
Not a production account system: add migrations, login/registration rate limits,
HTTPS, account recovery/verification, operational controls, and a reviewed
identity/token lifecycle before real deployment. No refresh/revocation service.
create_all is a classroom bootstrap, not a migration system.
"""
(project_dir / "README.md").write_text(readme, encoding="utf-8")
print("Generated project:", project_dir.resolve())
print("Files:", ", ".join(sorted(p.name for p in project_dir.iterdir() if p.is_file())))

Generated project: /workspace/scratch/d2095f4da170/fastapi_task_project
Files: .dockerignore, .env.example, .gitignore, Dockerfile, README.md, main.py, requirements.txt, test_api.py


## 54. Execute the full project tests

The tests use temporary SQLite databases; they do not alter a live application database. They verify persistence, validation, duplicate usernames, bad credentials/tokens, pagination, null handling, and cross-user access. The tests use synthetic credentials and never print tokens.

In [48]:
import subprocess
run = subprocess.run([sys.executable, "-m", "pytest", "-q", "--disable-warnings"],
                     cwd=project_dir, capture_output=True, text=True, timeout=180)
print(run.stdout)
if run.returncode:
    print(run.stderr)
assert run.returncode == 0, "Capstone tests failed; inspect output above."
print("Capstone integration tests passed.")

....................                                                     [100%]
20 passed, 2 warnings in 5.36s

Capstone integration tests passed.


# Checks and Teaching Activities
## Guided labs
| Lab | Assignment | Acceptance criteria |
|---|---|---|
| 1 | Add typed GET routes and filters | Valid request succeeds; invalid bounds → 422 |
| 2 | Build nested Pydantic order schemas | Required/null/extra-field tests pass |
| 3 | Add a cross-field delivery-date rule | Invalid ranges fail without database calls |
| 4 | Add router/service modules | HTTP behavior unchanged after refactor |
| 5 | Add task priority filtering and sorting | Allowed values validated; stable order; user isolation |
| 6 | Add a due date | Define date/time policy; test omitted vs null |
| 7 | Add an admin-only operation | Deny unauthenticated/non-admin callers; owner rules stay intact |
| 8 | Add a database migration | Upgrade a copy; verify data preserved |
| 9 | Add a job status resource | 202 + status URL; failures/retries observable |
| 10 | Package and review deployment | Secrets external; migrations/probes documented |

## Capstone extensions
1. Split the single-module app into routers, schemas, services, database, and security modules.
2. Add created/updated timestamps using a declared UTC convention; test updates and serialization.
3. Add search with bounded length and safe query construction. Explain wildcard escaping and indexing.
4. Implement task ownership transfer only under an explicit authorization policy.
5. Add optimistic concurrency/version checks; demonstrate conflicting updates return 409 or another documented conflict contract.
6. Add keyset pagination with a stable order; test inserts between page requests.
7. Replace local identity with a mockable provider-verification dependency and test issuer/audience failures.
8. Add a migration and a PostgreSQL test environment; compare transaction behavior with SQLite.

## Assessment rubric
| Criterion | Points |
|---|---:|
| Correct HTTP/resource design | 10 |
| Pydantic contracts and validators | 20 |
| CRUD and transaction behavior | 15 |
| Authentication and object-level authorization | 20 |
| Meaningful tests and isolated fixtures | 20 |
| Structure, documentation, and operational plan | 15 |
| **Total** | **100** |

## Oral questions with instructor cues
- **Does a type hint validate ordinary Python calls?** No; a runtime validator must enforce it.
- **Is nullable the same as optional?** No; null acceptance and field omission are separate.
- **Why avoid a database query inside a Pydantic validator?** Separate transport parsing from I/O/business rules; keep concurrency correctness in database constraints.
- **Why can a signed JWT still expose data?** Signing provides integrity, not confidentiality.
- **Why can valid authentication still permit a data breach?** Missing owner/tenant authorization.
- **Why is async not always faster?** It helps suitable concurrent I/O; blocking/CPU work still consumes execution resources.
- **Why not share one Session globally?** Sessions represent units of work and must not be shared across concurrent requests/tasks.
- **Why does PATCH need exclude_unset?** To preserve omitted fields while applying explicit changes.
- **Why does background work need its own session?** Request-scoped resources can close before the work completes.
- **What does TestClient not prove?** Real network/TLS/proxy behavior, capacity, and durable job guarantees.

## Troubleshooting setup
Use the interpreter/kernel where packages were installed; print `sys.executable` to check. Restart after upgrading packages. If a port is occupied, choose another for the optional server. If JWT settings are missing, supply a generated environment secret before invoking the app factory. If project tests fail after editing code, read the first failure and rerun only that test while debugging, then rerun the suite.

# Additional Pydantic and FastAPI Reference
## Specialized features to recognize
| Feature | Appropriate use | Important caveat |
|---|---|---|
| Pydantic dataclasses | Validated dataclass-style objects | Different APIs from BaseModel; use TypeAdapter for schema/serialization |
| Recursive models / forward annotations | Trees and linked schemas | Cycles and excessive nesting require bounds |
| Validation context | Pass caller-independent validation policy | Do not turn client-controlled context into authorization |
| `validate_call(validate_return=True)` | Function input and return contracts | Adds runtime overhead; keep boundaries purposeful |
| Strict types / constrained annotations | Explicit coercion policy | Python and JSON strict validation can differ |
| `Json[T]` | JSON text embedded in another format | Distinguish a JSON string from an already parsed object |
| Private attributes | Internal model state | Not a secret-management system |
| `model_rebuild()` | Resolve forward references when needed | Usually automatic for straightforward models |
| Custom types / core schema | Reusable specialist validation | Advanced API; prefer ordinary types and validators first |
| Custom response errors | Stable client-facing error contract | Avoid leaking input/validation internals |
| Mounted sub-applications | Independent app boundaries | Prefix/routing/schema/lifecycle behavior needs tests |
| Static files and templates | Serve frontend assets/HTML | Escape user content; avoid unsafe path resolution |
| OAuth scopes with `Security` | Declare and enforce scope requirements | Documentation alone does not enforce your policy |
| `Request`/raw `Response` | Low-level protocol access | Bypassing normal parsing/serialization is your responsibility |

The remaining reference example demonstrates recursive models, validation context, and dataclasses. Library internals and provider-specific services are outside this notebook's core scope.

In [49]:
from pydantic import ValidationInfo
from pydantic.dataclasses import dataclass as validated_dataclass
class TreeNode(BaseModel):
    name: str
    children: list["TreeNode"] = Field(default_factory=list)
root = TreeNode.model_validate({"name": "root", "children": [{"name": "leaf"}]})
assert root.children[0].name == "leaf"
class PolicyValue(BaseModel):
    quantity: int
    @field_validator("quantity")
    @classmethod
    def policy_limit(cls, value: int, info: ValidationInfo):
        maximum = (info.context or {}).get("maximum", 100)
        if value < 0 or value > maximum:
            raise ValueError("Quantity outside policy")
        return value
assert PolicyValue.model_validate({"quantity": 3}, context={"maximum": 5}).quantity == 3
@validated_dataclass
class Coordinate:
    x: float
    y: float
point = TypeAdapter(Coordinate).validate_python({"x": "1.5", "y": 2})
print("Recursive, context, and dataclass examples passed:", point)

Recursive, context, and dataclass examples passed: Coordinate(x=1.5, y=2.0)


## Official references and continuing study
The notebook uses original teaching explanations and examples informed by these primary sources. Documentation links were checked during preparation (13 November 2026); use the installed-version documentation when behavior differs.

- [FastAPI tutorial](https://fastapi.tiangolo.com/tutorial/)
- [FastAPI testing](https://fastapi.tiangolo.com/tutorial/testing/)
- [FastAPI dependencies](https://fastapi.tiangolo.com/tutorial/dependencies/)
- [FastAPI lifespan](https://fastapi.tiangolo.com/advanced/events/)
- [FastAPI OAuth2/JWT teaching example](https://fastapi.tiangolo.com/tutorial/security/oauth2-jwt/)
- [FastAPI SQL database guide](https://fastapi.tiangolo.com/tutorial/sql-databases/)
- [Pydantic models](https://docs.pydantic.dev/latest/concepts/models/)
- [Pydantic validators](https://docs.pydantic.dev/latest/concepts/validators/)
- [Pydantic serialization](https://docs.pydantic.dev/latest/concepts/serialization/)
- [Pydantic Settings](https://docs.pydantic.dev/latest/concepts/pydantic_settings/)
- [SQLAlchemy session basics](https://docs.sqlalchemy.org/en/20/orm/session_basics.html)

## Next steps
Teach the small examples before the capstone. Ask learners to predict each request's response, run it, then explain the result. Review the generated project with the acceptance criteria and require a demonstration of both successful and denied access.

**Validation scope:** all Python cells and the generated pytest suite are executed during preparation. Optional terminal/server/container/migration/cloud snippets are instructional and are not deployed by this notebook.

In [50]:
# Runtime record: useful when comparing a learner's environment with the tested course.
print("Interpreter:", sys.version.split()[0])
for name in ["fastapi", "starlette", "pydantic", "pydantic-settings", "sqlalchemy", "httpx", "PyJWT", "pwdlib", "pytest"]:
    print(f"{name}=={version(name)}")
print("All teaching cells reached the final validation record.")

Interpreter: 3.12.14
fastapi==0.141.1
starlette==1.6.0
pydantic==2.13.4
pydantic-settings==2.15.0
sqlalchemy==2.1.8
httpx==0.28.1
PyJWT==2.13.0
pwdlib==0.3.1
pytest==9.2.0
All teaching cells reached the final validation record.


## Preparation verification
All 50 Python cells executed sequentially, including top-level await; the generated capstone suite passed **20 tests**. A normal Jupyter kernel launch was blocked by this build environment’s network-interface restrictions, so execution used an in-process Python runner. Outputs are saved in this notebook. To verify in your own Jupyter environment, use **Restart & Run All** after installing dependencies. Live-server, Docker, cloud deployment, and migration steps were not run.

An HTML preview was generated, but browser-based visual inspection was unavailable because the build environment blocks browser sockets. Notebook structure, code compilation, saved outputs, and tests were checked programmatically.